<a href="https://colab.research.google.com/github/poojithparchuri-iiitk/Human-Activity-Recognition-using-UCI-HAR-ViT-timeseries-images/blob/main/UCI_HAR_signal_to_images_GADF%20%5B%20vannilla%20ViT%20%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyts

In [ ]:
!unzip data.zip

In [ ]:
import numpy as np
import os

DATA_DIR = "/content/UCI HAR Dataset"

def load_signal(file):
    return np.loadtxt(file)

def load_group(files):
    data = [load_signal(os.path.join(DATA_DIR, f)) for f in files]
    return np.stack(data, axis=1)

train_files = [
    "train/Inertial Signals/body_acc_x_train.txt",
    "train/Inertial Signals/body_acc_y_train.txt",
    "train/Inertial Signals/body_acc_z_train.txt",
    "train/Inertial Signals/body_gyro_x_train.txt",
    "train/Inertial Signals/body_gyro_y_train.txt",
    "train/Inertial Signals/body_gyro_z_train.txt",
]

test_files = [
    "test/Inertial Signals/body_acc_x_test.txt",
    "test/Inertial Signals/body_acc_y_test.txt",
    "test/Inertial Signals/body_acc_z_test.txt",
    "test/Inertial Signals/body_gyro_x_test.txt",
    "test/Inertial Signals/body_gyro_y_test.txt",
    "test/Inertial Signals/body_gyro_z_test.txt",
]

X_train_raw = load_group(train_files)
X_test_raw  = load_group(test_files)

print("Raw shapes:", X_train_raw.shape, X_test_raw.shape)

In [ ]:
import numpy as np

def gadf_transform(signal):
    x = (signal - signal.min()) / (signal.max() - signal.min() + 1e-8)
    x = 2 * x - 1
    phi = np.arccos(x)
    gadf = np.sin(phi[:, None] - phi[None, :])
    return gadf.astype(np.float32)


In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset

class HAR_GADF_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        sample = self.X[i]
        chans = []

        for axis in sample:
            gadf = gadf_transform(axis)
            chans.append(gadf)

        x = np.stack(chans, axis=0)
        x = torch.from_numpy(x).float()
        y = torch.tensor(self.y[i]).long()
        return x, y


In [ ]:
from torch.utils.data import DataLoader
import os
import numpy as np

y_train_path = os.path.join(DATA_DIR, "train/y_train.txt")
y_test_path = os.path.join(DATA_DIR, "test/y_test.txt")

y_train = load_signal(y_train_path).astype(np.int32) - 1
y_test = load_signal(y_test_path).astype(np.int32) - 1
train_dl = DataLoader(
    HAR_GADF_Dataset(X_train_raw, y_train),
    batch_size=32, shuffle=True, num_workers=0, pin_memory=True
)

test_dl = DataLoader(
    HAR_GADF_Dataset(X_test_raw, y_test),
    batch_size=32, shuffle=False, num_workers=0, pin_memory=True
)

In [ ]:
x, y = next(iter(train_dl))
print(x.shape, y.shape)

In [ ]:
x, y = next(iter(train_dl))
print("Batch X:", x.shape)
print("Batch Y:", y.shape)

In [ ]:
import torch
import torch.nn as nn
import timm
import math


class ViT6Pretrained(nn.Module):
    def __init__(self, num_classes, image_size=128, patch_size=16):
        super().__init__()

        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=num_classes
        )

        old = self.vit.patch_embed.proj
        self.vit.patch_embed.proj = nn.Conv2d(
            in_channels=6,
            out_channels=old.out_channels,
            kernel_size=old.kernel_size,
            stride=old.stride,
            padding=old.padding,
            bias=old.bias is not None
        )

        with torch.no_grad():
            self.vit.patch_embed.proj.weight[:, :3] = old.weight
            self.vit.patch_embed.proj.weight[:, 3:] = 0.0

        pos = self.vit.pos_embed[:, 1:, :]
        num_patches = (image_size // patch_size) ** 2
        pos = pos.reshape(1, int(math.sqrt(pos.shape[1])), int(math.sqrt(pos.shape[1])), -1)
        pos = torch.nn.functional.interpolate(
            pos.permute(0, 3, 1, 2),
            size=(image_size // patch_size, image_size // patch_size),
            mode="bicubic",
            align_corners=False
        ).permute(0, 2, 3, 1).reshape(1, num_patches, -1)

        cls_tok = self.vit.pos_embed[:, :1, :]
        self.vit.pos_embed = nn.Parameter(torch.cat([cls_tok, pos], dim=1))
        self.vit.patch_embed.img_size = (image_size, image_size)
        self.vit.patch_embed.strict_img_size = False
        self.vit.img_size = image_size


    def forward(self, x):
        return self.vit(x)

In [ ]:
device = "cuda"
num_classes = len(set(y_train))

model =ViT6Pretrained(num_classes).to(device)
print("Model ready ✔")

In [ ]:
for name, module in model.named_children():
    print(name, " -> ", type(module))

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for p in model.parameters():
    p.requires_grad = False

for p in model.vit.patch_embed.parameters():
    p.requires_grad = True

for p in model.vit.blocks[-2:].parameters():
    p.requires_grad = True

for p in model.vit.head.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {"params": model.vit.patch_embed.parameters(), "lr": 1e-4},
        {"params": model.vit.blocks[-2:].parameters(), "lr": 5e-5},
        {"params": model.vit.head.parameters(), "lr": 1e-3},
    ],
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

criterion = nn.CrossEntropyLoss()

print("DeepViT fine-tuning setup ready (patch_embed + last blocks + head)")

In [ ]:
from tqdm.auto import tqdm
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

EPOCHS = 16

for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for x, y in loop:
        x = x.to(device, non_blocking=True)
        y = y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type = "cuda"):
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loop.set_postfix(loss=float(loss))

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_dl:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    print("Val Acc:", correct / total)

In [ ]:
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for x, y in test_dl:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        preds = logits.argmax(1)

        test_correct += (preds == y).sum().item()
        test_total   += y.size(0)

test_acc = test_correct / test_total
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for x, y in test_dl:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        probs = torch.softmax(logits, dim=1)

        preds = probs.argmax(1)

        all_preds.append(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())

all_preds  = np.concatenate(all_preds)
all_probs  = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

acc = accuracy_score(all_labels, all_preds)

prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="macro", zero_division=0
)

prec_weight, rec_weight, f1_weight, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="weighted", zero_division=0
)
try:
    auroc_macro = roc_auc_score(
        all_labels, all_probs,
        multi_class="ovr",
        average="macro"
    )
except Exception:
    auroc_macro = None

print("GADF + Patch Embedding + finetuning:")
print("\n===== TEST METRICS =====")
print(f"Accuracy        : {acc:.4f}")
print(f"Precision (macro): {prec_macro:.4f}")
print(f"Recall (macro)   : {rec_macro:.4f}")
print(f"F1-Score (macro) : {f1_macro:.4f}")
print(f"Precision (weighted): {prec_weight:.4f}")
print(f"Recall (weighted)   : {rec_weight:.4f}")
print(f"F1-Score (weighted) : {f1_weight:.4f}")

if auroc_macro is not None:
    print(f"AUROC (macro-OvR): {auroc_macro:.4f}")
else:
    print("AUROC: not defined (check class probabilities)")

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(model, test_loader):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for x, y in test_loader:

            x = x.to(device)

            outputs = model(x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            y_true.extend(y.numpy())
            y_pred.extend(preds)


    cm = confusion_matrix(y_true, y_pred, normalize="true")
    labels = [
        "WALKING",
        "WALK_UP",
        "WALK_DOWN",
        "SITTING",
        "STANDING",
        "LAYING"
    ]

    plt.figure(figsize=(8, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Normalized Confusion Matrix (GADF - UCI HAR)")
    plt.show()
plot_confusion_matrix(model, test_dl)

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

def activity_wise_table(model, test_loader):

    model.eval()

    y_true = []
    y_pred = []
    with torch.no_grad():
        for x, y in test_loader:

            x = x.to(device)

            outputs = model(x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            y_true.extend(y.numpy())
            y_pred.extend(preds)
    class_names = [
        "Walking",
        "Walking Upstairs",
        "Walking Downstairs",
        "Sitting",
        "Standing",
        "Laying"
    ]
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True
    )

    cm = confusion_matrix(y_true, y_pred)
    table = []

    for i, activity in enumerate(class_names):
        precision = report[activity]["precision"]
        recall    = report[activity]["recall"]
        f1        = report[activity]["f1-score"]
        acc = cm[i, i] / cm[i].sum()

        table.append([
            activity,
            precision * 100,
            recall * 100,
            f1 * 100,
            acc * 100
        ])
    df = pd.DataFrame(
        table,
        columns=["Activity", "Precision (%)", "Recall (%)", "F1-score (%)", "Accuracy (%)"]
    )
    print("\nActivity-wise Performance Table (UCI-HAR)\n")
    display(df)
activity_wise_table(model, test_dl)